In [ ]:
import math
from typing import Dict, List, Tuple

import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    set_seed,
    )
from vllm import LLM, SamplingParams

# ======================
# Configuration
# ======================
BASE_MODEL = "august66/qwen2.5-1.5b-base-hh-helpful-sft"
PROMPT_DATASET = "august66/hh_helpfulness_drpo_from_sft"
REWARD_MODEL = "Kyleyee/Qwen2.5-1.5B-reward-hh-retrain"

# Cache paths (assumed to already exist)
DATA_CACHE_DIR = "/root/autodl-tmp/data_cache"
MODEL_CACHE_DIR = "/root/autodl-tmp/model_cache"

# Split / size controls
SPLIT = "train"
MAX_SAMPLES = None  # set int (e.g. 1000) for a quick subset

# Generation controls (2 responses per prompt)
GEN_MAX_NEW_TOKENS = 512
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95
GEN_SEED = 42

# vLLM runtime controls
GEN_TENSOR_PARALLEL_SIZE = 1
GEN_DTYPE = "bfloat16"  # "auto", "float16", "bfloat16"
VLLM_ENFORCE_EAGER = True
VLLM_GPU_MEMORY_UTILIZATION = 0.70  # leave headroom for sampling/activations
VLLM_MAX_MODEL_LEN = 2048           # cap KV cache (Qwen2.5 default 131072 is way too large)
VLLM_MAX_NUM_SEQS = 64              # limit concurrent sequences to avoid OOM during sampling

# Fallback controls
FALLBACK_TO_TRANSFORMERS_IF_VLLM_FAILS = True
FALLBACK_BATCH_SIZE = 1

# Reward model runtime
REWARD_BATCH_SIZE = 16

# Output path
OUTPUT_DATASET_DIR = "./hh_helpfulness_bt_oracle_dataset"

set_seed(GEN_SEED)

print("Config loaded.")
print(f"Base model:      {BASE_MODEL}")
print(f"Prompt data:     {PROMPT_DATASET} ({SPLIT})")
print(f"Reward model:    {REWARD_MODEL}")
print(f"Data cache dir:  {DATA_CACHE_DIR}")
print(f"Model cache dir: {MODEL_CACHE_DIR}")
print(f"vLLM eager:      {VLLM_ENFORCE_EAGER}")
print(f"vLLM max_model_len: {VLLM_MAX_MODEL_LEN}")
print(f"vLLM max_num_seqs:  {VLLM_MAX_NUM_SEQS}")
print(f"vLLM gpu_mem_util:  {VLLM_GPU_MEMORY_UTILIZATION}")
print(f"Fallback on:     {FALLBACK_TO_TRANSFORMERS_IF_VLLM_FAILS}")

In [ ]:
import json
from tqdm.auto import tqdm


# ── helpers ──────────────────────────────────────────────────────────

def _inspect_dataset(raw: Dataset, num_rows: int = 3) -> None:
    print("Dataset inspection:")
    print(f"- num rows: {len(raw)}")
    print(f"- columns:  {raw.column_names}")
    print(f"- features: {raw.features}")
    for i in range(min(num_rows, len(raw))):
        row = raw[i]
        print(f"\nSample row {i}:")
        for key in raw.column_names:
            val = row[key]
            preview = repr(val)
            if len(preview) > 200:
                preview = preview[:200] + "..."
            print(f"  - {key}: {preview}")


def _extract_prompt_messages(raw: Dataset) -> List[List[dict]]:
    """Extract prompts as structured conversation message lists.

    Each prompt is kept as List[dict] with 'role' and 'content' keys,
    preserving the multi-turn conversation structure.
    """
    all_messages: List[List[dict]] = []
    for row in raw:
        prompt = row["prompt"]
        if isinstance(prompt, list):
            messages = []
            for item in prompt:
                if isinstance(item, dict) and "role" in item and "content" in item:
                    messages.append({
                        "role": str(item["role"]).strip(),
                        "content": str(item["content"]).strip(),
                    })
            all_messages.append(messages)
        elif isinstance(prompt, str):
            all_messages.append([{"role": "user", "content": prompt.strip()}])
        else:
            all_messages.append([{"role": "user", "content": str(prompt).strip()}])
    return all_messages


def _format_for_generation(tokenizer, messages: List[dict]) -> str:
    """Apply chat template to a multi-turn conversation for generation.

    Uses add_generation_prompt=True so the model will produce only
    the next assistant turn.
    """
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        except Exception:
            pass
    # Fallback: manual formatting
    parts = []
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        parts.append(f"{'User' if role == 'user' else 'Assistant'}: {content}")
    parts.append("Assistant:")
    return "\n".join(parts)


def _build_scored_text(tokenizer, messages: List[dict], response: str) -> str:
    """Build full conversation + final assistant response for reward scoring."""
    full_messages = messages + [{"role": "assistant", "content": response}]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                full_messages, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            pass
    parts = []
    for msg in full_messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        parts.append(f"{'User' if role == 'user' else 'Assistant'}: {content}")
    return "\n".join(parts)


def _build_stop_token_ids(tokenizer) -> List[int]:
    """Collect stop token IDs: eos + Qwen special end tokens."""
    stop_ids = []
    if tokenizer.eos_token_id is not None:
        stop_ids.append(tokenizer.eos_token_id)
    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if isinstance(tid, int) and tid != tokenizer.unk_token_id:
            stop_ids.append(tid)
    # deduplicate while preserving order
    seen = set()
    unique = []
    for sid in stop_ids:
        if sid not in seen:
            seen.add(sid)
            unique.append(sid)
    return unique


# ── vLLM generation ─────────────────────────────────────────────────

def generate_two_responses_with_vllm(
    prompt_messages: List[List[dict]],
) -> Tuple[List[str], List[str]]:
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR
    )
    gen_texts = [_format_for_generation(tokenizer, msgs) for msgs in prompt_messages]
    stop_token_ids = _build_stop_token_ids(tokenizer)
    print(f"Stop token IDs: {stop_token_ids}")

    llm = LLM(
        model=BASE_MODEL,
        tensor_parallel_size=GEN_TENSOR_PARALLEL_SIZE,
        dtype=GEN_DTYPE,
        trust_remote_code=True,
        enforce_eager=VLLM_ENFORCE_EAGER,
        gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
        max_model_len=VLLM_MAX_MODEL_LEN,
        max_num_seqs=VLLM_MAX_NUM_SEQS,
        download_dir=MODEL_CACHE_DIR,
    )
    sampling_params = SamplingParams(
        n=2,
        max_tokens=GEN_MAX_NEW_TOKENS,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        seed=GEN_SEED,
        stop_token_ids=stop_token_ids,
    )

    generations = llm.generate(gen_texts, sampling_params)

    response_1, response_2 = [], []
    for item in generations:
        outs = [o.text.strip() for o in item.outputs]
        outs += [""] * (2 - len(outs))
        response_1.append(outs[0])
        response_2.append(outs[1])
    return response_1, response_2


# ── transformers fallback (batched) ─────────────────────────────────

@torch.no_grad()
def generate_two_responses_with_transformers(
    prompt_messages: List[List[dict]],
) -> Tuple[List[str], List[str]]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR
    )
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"

    stop_token_ids = _build_stop_token_ids(tokenizer)
    print(f"Stop token IDs: {stop_token_ids}")

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, trust_remote_code=True, dtype=dtype, cache_dir=MODEL_CACHE_DIR,
    ).to(device).eval()

    gen_texts = [_format_for_generation(tokenizer, msgs) for msgs in prompt_messages]

    response_1, response_2 = [], []
    batch_size = max(FALLBACK_BATCH_SIZE, 1)

    for start in tqdm(range(0, len(gen_texts), batch_size), desc="Generating (transformers)"):
        batch = gen_texts[start : start + batch_size]
        encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)

        outputs = model.generate(
            **encoded,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            num_return_sequences=2,
            eos_token_id=stop_token_ids,
            pad_token_id=tokenizer.pad_token_id,
        )
        input_len = encoded["input_ids"].shape[1]
        decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

        for i in range(len(batch)):
            r1 = decoded[i * 2].strip() if i * 2 < len(decoded) else ""
            r2 = decoded[i * 2 + 1].strip() if i * 2 + 1 < len(decoded) else ""
            response_1.append(r1)
            response_2.append(r2)

    return response_1, response_2


# ── reward scoring (loaded ONCE, scores both response sets) ─────────

@torch.no_grad()
def score_all_responses(
    prompt_messages: List[List[dict]],
    response_1: List[str],
    response_2: List[str],
    batch_size: int = 16,
    max_length: int = 2048,
) -> Tuple[List[float], List[float]]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(
        REWARD_MODEL, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        REWARD_MODEL, trust_remote_code=True, dtype=dtype, cache_dir=MODEL_CACHE_DIR,
    ).to(device).eval()

    # Build ALL texts at once: [r1_0, r2_0, r1_1, r2_1, ...]
    all_texts: List[str] = []
    for msgs, r1, r2 in zip(prompt_messages, response_1, response_2):
        all_texts.append(_build_scored_text(tokenizer, msgs, r1))
        all_texts.append(_build_scored_text(tokenizer, msgs, r2))

    all_scores: List[float] = []
    for start in tqdm(range(0, len(all_texts), batch_size), desc="Reward scoring"):
        batch = all_texts[start : start + batch_size]
        encoded = tokenizer(
            batch, return_tensors="pt", truncation=True,
            max_length=max_length, padding=True,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        logits = model(**encoded).logits

        if logits.ndim == 2 and logits.shape[-1] == 1:
            scores = logits[:, 0]
        elif logits.ndim == 2:
            scores = logits[:, -1]
        else:
            scores = logits.squeeze(-1)
        all_scores.extend(scores.float().cpu().tolist())

    # De-interleave
    scores_1 = all_scores[0::2]
    scores_2 = all_scores[1::2]
    return scores_1, scores_2


# ── BT oracle preference ────────────────────────────────────────────

def bt_probability(score_a: float, score_b: float) -> Tuple[float, float]:
    diff = max(min(score_a - score_b, 60.0), -60.0)
    p_a = 1.0 / (1.0 + math.exp(-diff))
    return p_a, 1.0 - p_a


# ── main pipeline ───────────────────────────────────────────────────

def build_bt_oracle_dataset() -> Dataset:
    raw = load_dataset(PROMPT_DATASET, split=SPLIT, cache_dir=DATA_CACHE_DIR)
    if MAX_SAMPLES is not None:
        raw = raw.select(range(min(MAX_SAMPLES, len(raw))))

    _inspect_dataset(raw, num_rows=2)

    prompt_messages = _extract_prompt_messages(raw)
    print(f"\nLoaded {len(prompt_messages)} prompts.")
    print(f"Example conversation ({len(prompt_messages[0])} turns):")
    for msg in prompt_messages[0]:
        content_preview = msg['content'][:120] + "..." if len(msg['content']) > 120 else msg['content']
        print(f"  [{msg['role']}]: {content_preview}")

    # ── Generation ──
    print("\nGenerating 2 responses per prompt...")
    generation_backend = "vllm"
    try:
        response_1, response_2 = generate_two_responses_with_vllm(prompt_messages)
    except Exception as err:
        if not FALLBACK_TO_TRANSFORMERS_IF_VLLM_FAILS:
            raise
        generation_backend = "transformers_fallback"
        print(f"vLLM failed ({type(err).__name__}), falling back to transformers.")
        response_1, response_2 = generate_two_responses_with_transformers(prompt_messages)
    print(f"Generation backend: {generation_backend}")

    # ── Reward scoring (single model load) ──
    print("\nScoring responses with reward model...")
    reward_1, reward_2 = score_all_responses(
        prompt_messages, response_1, response_2, batch_size=REWARD_BATCH_SIZE
    )

    # ── BT oracle preference ──
    bt_prob_1, bt_prob_2 = [], []
    chosen, rejected, chosen_idx = [], [], []

    for r1, r2, s1, s2 in zip(response_1, response_2, reward_1, reward_2):
        p1, p2 = bt_probability(s1, s2)
        bt_prob_1.append(p1)
        bt_prob_2.append(p2)
        if p1 >= p2:
            chosen.append(r1); rejected.append(r2); chosen_idx.append(0)
        else:
            chosen.append(r2); rejected.append(r1); chosen_idx.append(1)

    return Dataset.from_dict({
        "prompt": prompt_messages,        # structured conversation (List[List[dict]])
        "response_1": response_1,          # final assistant response only
        "response_2": response_2,
        "reward_1": reward_1,
        "reward_2": reward_2,
        "bt_prob_response_1": bt_prob_1,
        "bt_prob_response_2": bt_prob_2,
        "chosen_idx": chosen_idx,
    })


bt_dataset = build_bt_oracle_dataset()
print(bt_dataset)

bt_dataset.save_to_disk(OUTPUT_DATASET_DIR)
print(f"Saved HF dataset to: {OUTPUT_DATASET_DIR}")

PUSH_TO_HUB = True
REPO_ID = 'august66/hh_helpfulness_bt_oracle_dataset'
if PUSH_TO_HUB:
    if REPO_ID is None:
        raise ValueError("Set REPO_ID before PUSH_TO_HUB=True")
    bt_dataset.push_to_hub(REPO_ID)
    print(f"Pushed dataset to hub: {REPO_ID}")

In [ ]:
# ── Post-process: wrap responses in conv format, re-score with prompt+response, re-push ──

import math
from typing import List, Tuple

import torch
from datasets import Dataset, load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from tqdm.auto import tqdm

# ── Config ──
HUB_DATASET = "august66/hh_helpfulness_bt_oracle_dataset"
REWARD_MODEL = "Kyleyee/Qwen2.5-1.5B-reward-hh-retrain"
DATA_CACHE_DIR = "/root/autodl-tmp/data_cache"
MODEL_CACHE_DIR = "/root/autodl-tmp/model_cache"
REWARD_BATCH_SIZE = 16
REWARD_MAX_LENGTH = 2048

# ── Load from Hub ──
ds = load_dataset(HUB_DATASET, split="train", cache_dir=DATA_CACHE_DIR)
print(f"Loaded {len(ds)} rows from {HUB_DATASET}")
print(f"Columns: {ds.column_names}")
print(f"Features: {ds.features}")


# ── Wrap raw response strings as conversation format ──
# response_1/response_2: just the assistant turn → [{"role": "assistant", "content": "..."}]
# NOT including prompt — prompt is stored separately.
resp_conv_1 = [[{"role": "assistant", "content": r}] for r in ds["response_1"]]
resp_conv_2 = [[{"role": "assistant", "content": r}] for r in ds["response_2"]]

print(f"\nWrapped responses in conversation format (assistant-only).")
print(f"  response_1[0]: {resp_conv_1[0]}")


# ── Re-evaluate rewards on prompt + response (full conversation) ──
def _format_for_reward(tokenizer, prompt_msgs: List[dict], response: str) -> str:
    """Format prompt + assistant response as full conversation for reward scoring."""
    full_conv = prompt_msgs + [{"role": "assistant", "content": response}]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                full_conv, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            pass
    parts = []
    for msg in full_conv:
        role = "User" if msg.get("role") == "user" else "Assistant"
        parts.append(f"{role}: {msg.get('content', '')}")
    return "\n".join(parts)


@torch.no_grad()
def score_with_prompt_and_response(
    prompts: List[List[dict]],
    responses: List[str],
    batch_size: int = 16,
    max_length: int = 2048,
) -> List[float]:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.bfloat16 if device == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(
        REWARD_MODEL, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        REWARD_MODEL, trust_remote_code=True, dtype=dtype, cache_dir=MODEL_CACHE_DIR,
    ).to(device).eval()

    # Format each as: prompt conversation + assistant response
    all_texts = [
        _format_for_reward(tokenizer, p, r) for p, r in zip(prompts, responses)
    ]

    scores: List[float] = []
    for start in tqdm(range(0, len(all_texts), batch_size), desc="Reward scoring"):
        batch = all_texts[start : start + batch_size]
        encoded = tokenizer(
            batch, return_tensors="pt", truncation=True,
            max_length=max_length, padding=True,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        logits = model(**encoded).logits

        if logits.ndim == 2 and logits.shape[-1] == 1:
            s = logits[:, 0]
        elif logits.ndim == 2:
            s = logits[:, -1]
        else:
            s = logits.squeeze(-1)
        scores.extend(s.float().cpu().tolist())

    del model
    torch.cuda.empty_cache()
    return scores


prompts = ds["prompt"]
raw_responses_1 = ds["response_1"]
raw_responses_2 = ds["response_2"]

print("\nScoring prompt + response_1 ...")
reward_1 = score_with_prompt_and_response(
    prompts, raw_responses_1, batch_size=REWARD_BATCH_SIZE, max_length=REWARD_MAX_LENGTH
)
print("Scoring prompt + response_2 ...")
reward_2 = score_with_prompt_and_response(
    prompts, raw_responses_2, batch_size=REWARD_BATCH_SIZE, max_length=REWARD_MAX_LENGTH
)


# ── Recompute BT oracle preference ──
def bt_probability(score_a: float, score_b: float) -> Tuple[float, float]:
    diff = max(min(score_a - score_b, 60.0), -60.0)
    p_a = 1.0 / (1.0 + math.exp(-diff))
    return p_a, 1.0 - p_a

bt_prob_1, bt_prob_2, chosen_idx = [], [], []
for s1, s2 in zip(reward_1, reward_2):
    p1, p2 = bt_probability(s1, s2)
    bt_prob_1.append(p1)
    bt_prob_2.append(p2)
    chosen_idx.append(0 if p1 >= p2 else 1)


# ── Build final dataset ──
final_ds = Dataset.from_dict({
    "prompt": prompts,              # multi-turn prompt (List[List[dict]])
    "response_1": resp_conv_1,      # assistant response only (List[dict]) — [{"role":"assistant","content":"..."}]
    "response_2": resp_conv_2,      # assistant response only (List[dict])
    "reward_1": reward_1,           # scored on full prompt + response conversation
    "reward_2": reward_2,
    "bt_prob_response_1": bt_prob_1,
    "bt_prob_response_2": bt_prob_2,
    "chosen_idx": chosen_idx,
})

# Sanity check
row = final_ds[0]
print(f"\nSanity check (row 0):")
print(f"  prompt turns:     {len(row['prompt'])}")
print(f"  response_1:       {row['response_1']}")
print(f"  response_2:       {row['response_2'][0]['content'][:100]}...")
print(f"  reward_1={row['reward_1']:.4f}  reward_2={row['reward_2']:.4f}")
print(f"  chosen_idx={row['chosen_idx']}")

print(f"\n{final_ds}")

# Push to hub
final_ds.push_to_hub(HUB_DATASET)
print(f"\nPushed updated dataset to: {HUB_DATASET}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MC Reward Sampling: Generate 8 responses per prompt from ref & target,
# score each with reward model, save as new dataset.
#
#   - Model 1 (ref):    august66/qwen2.5-1.5b-base-hh-helpful-sft
#   - Model 2 (target): august66/hh_qwen1.5_drpo_fixed_beta
#   - Prompts:          august66/hh_helpfulness_drpo_from_sft
#   - Reward:           Kyleyee/Qwen2.5-1.5B-reward-hh-retrain
#   - Output:           16 reward columns (ref_reward_0..7, target_reward_0..7)
# ═══════════════════════════════════════════════════════════════════════
import gc, math
from typing import List, Tuple

import numpy as np
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    set_seed,
)
from vllm import LLM, SamplingParams
from tqdm.auto import tqdm

# ── Config ──
REF_MODEL    = "august66/qwen2.5-1.5b-base-hh-helpful-sft"
TARGET_MODEL = "august66/hh_qwen1.5_drpo_fixed_beta"
PROMPT_DATASET = "august66/hh_helpfulness_drpo_from_sft"
REWARD_MODEL   = "Kyleyee/Qwen2.5-1.5B-reward-hh-retrain"

DATA_CACHE_DIR  = "/root/autodl-tmp/data_cache"
MODEL_CACHE_DIR = "/root/autodl-tmp/model_cache"

NUM_SAMPLES     = 8          # MC samples per prompt per model
GEN_MAX_NEW_TOKENS = 512
GEN_TEMPERATURE = 1.0
GEN_TOP_P       = 0.95
GEN_SEED        = 42

VLLM_DTYPE      = "bfloat16"
VLLM_ENFORCE_EAGER = True
VLLM_GPU_MEM_UTIL  = 0.70
VLLM_MAX_MODEL_LEN = 2048
VLLM_MAX_NUM_SEQS  = 64
VLLM_TP_SIZE       = 1

REWARD_BATCH_SIZE  = 16
REWARD_MAX_LENGTH  = 2048

SPLIT = "train"
MAX_SAMPLES = None  # set int for quick test

HUB_OUTPUT = "august66/hh_helpfulness_mc_rewards"

set_seed(GEN_SEED)
print("MC Reward Sampling config loaded.")
print(f"  Ref model:    {REF_MODEL}")
print(f"  Target model: {TARGET_MODEL}")
print(f"  Prompts:      {PROMPT_DATASET}")
print(f"  Reward model: {REWARD_MODEL}")
print(f"  MC samples:   {NUM_SAMPLES} per model per prompt")
print(f"  Output:       {HUB_OUTPUT}")


# ── Helpers (reuse from cell 2) ──

def _extract_prompt_messages(raw):
    all_messages = []
    for row in raw:
        prompt = row["prompt"]
        if isinstance(prompt, list):
            messages = []
            for item in prompt:
                if isinstance(item, dict) and "role" in item and "content" in item:
                    messages.append({"role": str(item["role"]).strip(),
                                     "content": str(item["content"]).strip()})
            all_messages.append(messages)
        elif isinstance(prompt, str):
            all_messages.append([{"role": "user", "content": prompt.strip()}])
        else:
            all_messages.append([{"role": "user", "content": str(prompt).strip()}])
    return all_messages


def _format_for_generation(tokenizer, messages):
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    parts = []
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        parts.append(f"{'User' if role == 'user' else 'Assistant'}: {content}")
    parts.append("Assistant:")
    return "\n".join(parts)


def _build_stop_token_ids(tokenizer):
    stop_ids = []
    if tokenizer.eos_token_id is not None:
        stop_ids.append(tokenizer.eos_token_id)
    for tok in ["<|im_end|>", "<|endoftext|>"]:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if isinstance(tid, int) and tid != tokenizer.unk_token_id:
            stop_ids.append(tid)
    seen = set()
    return [s for s in stop_ids if not (s in seen or seen.add(s))]


def _build_scored_text(tokenizer, messages, response):
    full = messages + [{"role": "assistant", "content": response}]
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            return tokenizer.apply_chat_template(full, tokenize=False, add_generation_prompt=False)
        except Exception:
            pass
    parts = []
    for msg in full:
        role = "User" if msg.get("role") == "user" else "Assistant"
        parts.append(f"{role}: {msg.get('content', '')}")
    return "\n".join(parts)


# ── Load prompts ──
raw = load_dataset(PROMPT_DATASET, split=SPLIT, cache_dir=DATA_CACHE_DIR)
if MAX_SAMPLES is not None:
    raw = raw.select(range(min(MAX_SAMPLES, len(raw))))
prompt_messages = _extract_prompt_messages(raw)
print(f"\nLoaded {len(prompt_messages)} prompts.")


# ═══════════════════════════════════════════
# Phase 1: Generate 8 responses from REF model
# ═══════════════════════════════════════════
print("\n" + "=" * 60)
print(f"Generating {NUM_SAMPLES} responses per prompt from REF model...")
print("=" * 60)

tokenizer_ref = AutoTokenizer.from_pretrained(
    REF_MODEL, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR)
gen_texts = [_format_for_generation(tokenizer_ref, msgs) for msgs in prompt_messages]
stop_ids = _build_stop_token_ids(tokenizer_ref)
print(f"Stop token IDs: {stop_ids}")

llm_ref = LLM(
    model=REF_MODEL,
    tensor_parallel_size=VLLM_TP_SIZE,
    dtype=VLLM_DTYPE,
    trust_remote_code=True,
    enforce_eager=VLLM_ENFORCE_EAGER,
    gpu_memory_utilization=VLLM_GPU_MEM_UTIL,
    max_model_len=VLLM_MAX_MODEL_LEN,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
    download_dir=MODEL_CACHE_DIR,
)
sampling_params = SamplingParams(
    n=NUM_SAMPLES,
    max_tokens=GEN_MAX_NEW_TOKENS,
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    seed=GEN_SEED,
    stop_token_ids=stop_ids,
)
ref_generations = llm_ref.generate(gen_texts, sampling_params)

# Collect: ref_responses[i] = list of NUM_SAMPLES strings
ref_responses = []
for item in ref_generations:
    outs = [o.text.strip() for o in item.outputs]
    outs += [""] * (NUM_SAMPLES - len(outs))
    ref_responses.append(outs[:NUM_SAMPLES])

del llm_ref
gc.collect()
torch.cuda.empty_cache()
print(f"Ref generation done. {len(ref_responses)} prompts × {NUM_SAMPLES} samples.")


# ═══════════════════════════════════════════
# Phase 2: Generate 8 responses from TARGET model
# ═══════════════════════════════════════════
print("\n" + "=" * 60)
print(f"Generating {NUM_SAMPLES} responses per prompt from TARGET model...")
print("=" * 60)

tokenizer_tgt = AutoTokenizer.from_pretrained(
    TARGET_MODEL, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR)
gen_texts_tgt = [_format_for_generation(tokenizer_tgt, msgs) for msgs in prompt_messages]
stop_ids_tgt = _build_stop_token_ids(tokenizer_tgt)
print(f"Stop token IDs: {stop_ids_tgt}")

llm_tgt = LLM(
    model=TARGET_MODEL,
    tensor_parallel_size=VLLM_TP_SIZE,
    dtype=VLLM_DTYPE,
    trust_remote_code=True,
    enforce_eager=VLLM_ENFORCE_EAGER,
    gpu_memory_utilization=VLLM_GPU_MEM_UTIL,
    max_model_len=VLLM_MAX_MODEL_LEN,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
    download_dir=MODEL_CACHE_DIR,
)
sampling_params_tgt = SamplingParams(
    n=NUM_SAMPLES,
    max_tokens=GEN_MAX_NEW_TOKENS,
    temperature=GEN_TEMPERATURE,
    top_p=GEN_TOP_P,
    seed=GEN_SEED,
    stop_token_ids=stop_ids_tgt,
)
tgt_generations = llm_tgt.generate(gen_texts_tgt, sampling_params_tgt)

tgt_responses = []
for item in tgt_generations:
    outs = [o.text.strip() for o in item.outputs]
    outs += [""] * (NUM_SAMPLES - len(outs))
    tgt_responses.append(outs[:NUM_SAMPLES])

del llm_tgt
gc.collect()
torch.cuda.empty_cache()
print(f"Target generation done. {len(tgt_responses)} prompts × {NUM_SAMPLES} samples.")


# ═══════════════════════════════════════════
# Phase 3: Score ALL responses with reward model
# ═══════════════════════════════════════════
print("\n" + "=" * 60)
print("Scoring all responses with reward model...")
print("=" * 60)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

reward_tokenizer = AutoTokenizer.from_pretrained(
    REWARD_MODEL, trust_remote_code=True, cache_dir=MODEL_CACHE_DIR)
reward_model = AutoModelForSequenceClassification.from_pretrained(
    REWARD_MODEL, trust_remote_code=True, dtype=dtype, cache_dir=MODEL_CACHE_DIR,
).to(device).eval()


@torch.no_grad()
def score_texts(texts: List[str], batch_size: int = 16, max_length: int = 2048) -> List[float]:
    scores = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Reward scoring"):
        batch = texts[start : start + batch_size]
        encoded = reward_tokenizer(
            batch, return_tensors="pt", truncation=True,
            max_length=max_length, padding=True,
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        logits = reward_model(**encoded).logits
        if logits.ndim == 2 and logits.shape[-1] == 1:
            s = logits[:, 0]
        elif logits.ndim == 2:
            s = logits[:, -1]
        else:
            s = logits.squeeze(-1)
        scores.extend(s.float().cpu().tolist())
    return scores


# Build flat list of texts: all ref samples then all target samples
n_prompts = len(prompt_messages)
all_texts = []
for i in range(n_prompts):
    for j in range(NUM_SAMPLES):
        all_texts.append(_build_scored_text(reward_tokenizer, prompt_messages[i], ref_responses[i][j]))
for i in range(n_prompts):
    for j in range(NUM_SAMPLES):
        all_texts.append(_build_scored_text(reward_tokenizer, prompt_messages[i], tgt_responses[i][j]))

print(f"Total texts to score: {len(all_texts)}  ({n_prompts} prompts × {NUM_SAMPLES} × 2 models)")
all_scores = score_texts(all_texts, batch_size=REWARD_BATCH_SIZE, max_length=REWARD_MAX_LENGTH)

# Reshape: first half = ref, second half = target
ref_scores_flat = all_scores[: n_prompts * NUM_SAMPLES]
tgt_scores_flat = all_scores[n_prompts * NUM_SAMPLES :]

# ref_rewards[i][j] = reward for prompt i, ref sample j
ref_rewards = [ref_scores_flat[i * NUM_SAMPLES : (i + 1) * NUM_SAMPLES] for i in range(n_prompts)]
tgt_rewards = [tgt_scores_flat[i * NUM_SAMPLES : (i + 1) * NUM_SAMPLES] for i in range(n_prompts)]

del reward_model
torch.cuda.empty_cache()

print(f"\nRef rewards:    mean={np.mean(ref_scores_flat):.4f}  std={np.std(ref_scores_flat):.4f}")
print(f"Target rewards: mean={np.mean(tgt_scores_flat):.4f}  std={np.std(tgt_scores_flat):.4f}")


# ═══════════════════════════════════════════
# Phase 4: Build dataset and push to Hub
# ═══════════════════════════════════════════
data_dict = {"prompt": prompt_messages}

for j in range(NUM_SAMPLES):
    data_dict[f"ref_reward_{j}"] = [ref_rewards[i][j] for i in range(n_prompts)]
for j in range(NUM_SAMPLES):
    data_dict[f"target_reward_{j}"] = [tgt_rewards[i][j] for i in range(n_prompts)]

mc_ds = Dataset.from_dict(data_dict)
print(f"\nMC reward dataset: {mc_ds}")
print(f"Columns: {mc_ds.column_names}")

# ── Sanity check: print a few sample responses (not saved) ──
NUM_PREVIEW = 3  # prompts to preview
NUM_RESP_PREVIEW = 2  # responses per model to show

for i in range(min(NUM_PREVIEW, n_prompts)):
    print(f"\n{'─' * 60}")
    print(f"Prompt {i} ({len(prompt_messages[i])} turns):")
    last_msg = prompt_messages[i][-1]
    content = last_msg['content']
    print(f"  [{last_msg['role']}]: {content[:150]}{'...' if len(content) > 150 else ''}")

    for j in range(min(NUM_RESP_PREVIEW, NUM_SAMPLES)):
        ref_resp = ref_responses[i][j][:120]
        tgt_resp = tgt_responses[i][j][:120]
        print(f"  ref  sample {j} (reward={ref_rewards[i][j]:+.4f}): {ref_resp}...")
        print(f"  tgt  sample {j} (reward={tgt_rewards[i][j]:+.4f}): {tgt_resp}...")

    ref_mean = np.mean(ref_rewards[i])
    tgt_mean = np.mean(tgt_rewards[i])
    print(f"  ref rewards:  {[f'{r:.3f}' for r in ref_rewards[i]]}")
    print(f"  tgt rewards:  {[f'{r:.3f}' for r in tgt_rewards[i]]}")
    print(f"  ref mean={ref_mean:.4f}  tgt mean={tgt_mean:.4f}")

print(f"\n{'─' * 60}")
print(f"Dataset: {mc_ds}")
print(f"Columns: {mc_ds.column_names}")

mc_ds.push_to_hub(HUB_OUTPUT)
print(f"\nPushed MC reward dataset to: {HUB_OUTPUT}")